# Phase 4 Showdown — A Walkthrough

**For a non-technical reader.** This notebook explains what we're doing, why it matters, and what the numbers mean. It assumes nothing about machine learning. Where it must use jargon, it explains it inline.

If you only read one paragraph: **we're trying to detect when a language model is about to give a wrong answer, by looking at the model's "thoughts" while it's reasoning, instead of waiting for the wrong answer to come out.** This notebook checks whether our detector works better than two existing alternatives, on three different model architectures, on a hard multi-step question-answering task.

Companion documents (the project's "trifecta"):

- [`_PLAN.md`](../_PLAN.md) — what we're trying to build, in phases.
- [`_AGENT.md`](../_AGENT.md) — chronological log of what we tried, what worked, what broke.
- [`_MODEL.md`](../_MODEL.md) — internal reference on the three models (architectures, chat templates, quirks).

## 1. The setup in plain English

A **large language model (LLM)** like ChatGPT generates text one word at a time. To do that, it runs a giant numerical computation called a **forward pass** for every word, and inside that computation it produces a vector of numbers called the **hidden state**. The hidden state is the model's internal "thought" right before it picks the next word.

When the model is reasoning through a hard problem (say, a multi-step question), it goes through many of these forward passes — sometimes hundreds. At some point in that chain, it might **start drifting** toward a wrong answer. The big idea this project tests:

> **The hidden state changes its character right before the model commits to a wrong answer. If we watch the hidden state at the right moments, we can tell *before the wrong word is generated* that the model is going off the rails.**

We call this the **latent gate** ("latent" = hidden, "gate" = a yes/no signal that says "stop and re-check").

We compare the latent gate against two other existing approaches that wait for the model to *say* something suspicious:

- **ReaLM-Retrieve simulator** — fires when the model uses uncertainty phrases like *"Let me verify..."*, *"Actually..."*, *"Hmm,"*.
- **GRIP simulator** — fires when there's a sudden spike in how confused the model is about its next word.
- **Random** baseline — fires randomly at the same rate, so we can tell if the others are doing better than chance.

The thesis of the paper is: **the latent gate fires *earlier* than the text-based alternatives, on every model architecture we test.** That's what this Showdown checks.

## 2. The three models — picked deliberately to bracket "the 2026 design space"

We don't test the latent gate on one model, because then a sceptical reader would say *"sure, it works on yours."* We test it on three models that span the spectrum of how 2026's frontier LLMs handle the speed/memory tradeoff. All three have **roughly the same compute per word** (~3-4 billion "active" parameters), but they get there in completely different ways:

| nickname | what it is | param count | attention design | weights on disk |
| --- | --- | --- | --- | --- |
| **Phi** | Microsoft Phi-4-mini-instruct | 3.8 B (dense) | Standard full attention everywhere | ~7 GB |
| **Gemma** | Google Gemma-4-26B-A4B-it | 25.2 B total / 3.8 B active per word | Mixture-of-Experts + sliding-window-then-global hybrid | ~49 GB |
| **Qwen** | Qwen 3.6-35B-A3B | 35 B total / ~3 B active per word | Mixture-of-Experts + 3:1 linear-attention hybrid (Gated DeltaNet + full attention) | ~67 GB |

**Why this matters:** if the latent gate works on all three, the claim is "it's architecture-agnostic" — not tied to one company's design choices. That's the headline.

Full architectural details, with every config field cross-checked against the live HuggingFace model cards, live in `_MODEL.md`.

## 3. The task — HotpotQA distractor

We give each model **200 multi-step questions** drawn from a public benchmark called **HotpotQA-distractor**. Each question comes with **10 paragraphs** of source material — only 2 are relevant; the other 8 are *distractors* that mention overlapping entities. The model has to read all 10, find the right two, and chain a reasoning step across them.

**Example question:**

> *"The director of the romantic comedy 'Big Stone Gap' is based in what New York city?"*
>
> *Gold answer:* "Greenwich Village, New York City"

The model has to find:
1. (Paragraph 9) "Big Stone Gap" was directed by Adriana Trigiani.
2. (Paragraph 3) Adriana Trigiani is based in Greenwich Village, New York City.

This is **multi-hop reasoning**: a single paragraph doesn't contain the answer; the model must connect two paragraphs.

**Why this dataset.** HotpotQA's structure ("multi-hop reasoning over distractors") is exactly the regime where a good model can still go wrong on a long chain of thought. That's where we expect the latent gate to help.

The 200 questions are **pre-split**: 120 for **calibration** (tuning the gate's sensitivity), 40 for **validation** (the actual numbers we report), 40 for **test** (sealed away for the paper write-up). This pre-split protocol is locked in by the Phase 4.4 leakage contract — see [`_PLAN.md`](../_PLAN.md) landmine **L5**.

## 4. The Phase 4.1 mistake (and why we re-ran as 4.1b)

We ran the first 200-question sweep on 2026-05-09. **It was unfair**, in a way that took a careful audit to spot.

Each of the three models has its own way of being prompted. We were treating them all the same, and that meant:

- **Qwen** was running in its **default thinking mode**: the chat template prefills `<think>\n` before the model starts generating, so the model thinks silently before answering. Thinking takes lots of tokens. Our 512-token cap hit **on every single question** — Qwen never finished thinking, never emitted an answer.
- **Gemma** was accidentally running in **non-thinking mode** (its template's default). This is the regime we wanted, but only by luck. EM = 57.5% (the only "honest" number in the table).
- **Phi** was running with our system instruction *stuffed into the user message*, plus pure greedy decoding. With long distractor contexts, Phi got into **repetition loops** — sentences like *"Have a great day ahead!"* repeated 4 times until the token cap.

So the first sweep wasn't measuring "how good are these models" — it was measuring "how much does each model's prompt protocol mismatch our driver."

**The fix (Option A).** Same 200 questions, same dataset split, but:
1. Add an explicit `system` role for all three.
2. Set `enable_thinking=False` for Qwen so it skips the silent reasoning.
3. Add a small repetition penalty (1.05) for Phi to break the loops.
4. Stop early when the model emits a clean `Answer: ...` line.

We verified each step against the live HuggingFace model cards (the assistant's training data is older than these models, so we trust the cards over memory). A small validator script — `scripts/04b_protocol_smoke.py` — runs three probes on each model on every new compute box and refuses to start the sweep if any model is misbehaving.

Full diagnosis in [`_AGENT.md` 2026-05-10 #11](../_AGENT.md).

## 5. Setup — load the data we'll be looking at

The cells below load the per-question records produced by the 4.1b sweep. Each record contains: the question, the gold answer, the model's answer, whether it was right (EM = exact match, F1 = partial credit), how many tokens it took, and — crucially for us — a per-step trace of the four gates' fire decisions plus the underlying hidden-state features.

In [1]:
import json
from pathlib import Path
from collections import Counter
import pandas as pd

ROOT = Path("..").resolve()
RUNS = ROOT / "runs"

MODELS = ["phi-4-mini-instruct", "gemma-4-26b-a4b-it", "qwen3.6-35b-a3b-instruct"]
NICE = {"phi-4-mini-instruct": "Phi", "gemma-4-26b-a4b-it": "Gemma", "qwen3.6-35b-a3b-instruct": "Qwen"}

def load(model):
    p = RUNS / f"phase4_qa_{model}_mid_stack_v2.jsonl"
    rows = [json.loads(l) for l in p.open() if l.strip()]
    return [r for r in rows if "error" not in r]

traces = {m: load(m) for m in MODELS}
for m, ts in traces.items():
    splits = Counter(t.get("split", "?") for t in ts)
    print(f"{NICE[m]:6}  total={len(ts):3}  splits={dict(splits)}")

Phi     total=200  splits={'calibration': 120, 'test': 40, 'validation': 40}
Gemma   total=200  splits={'calibration': 120, 'test': 40, 'validation': 40}
Qwen    total=200  splits={'calibration': 120, 'test': 40, 'validation': 40}


### 5a. Headline accuracy — did the protocol fix help?

Before any gate analysis, the simplest sanity check: is each model now answering questions correctly at all? We compare 4.1b (Option A) against the original 4.1 numbers stored in `_AGENT.md`.

In [2]:
def stats(ts):
    n = len(ts)
    em = sum(t["em"] for t in ts) / n
    f1 = sum(t["f1"] for t in ts) / n
    tok = sum(t["tokens_generated"] for t in ts) / n
    steps = sum(t["n_steps"] for t in ts) / n
    return dict(N=n, EM=round(em, 3), F1=round(f1, 3), avg_tokens=round(tok, 1), avg_steps=round(steps, 1))

# Old 4.1 numbers from _AGENT.md (rejected sweep, kept here for contrast)
old = {
    "Phi":   dict(EM=0.265, F1=0.378, avg_tokens=402.6),
    "Gemma": dict(EM=0.575, F1=0.682, avg_tokens=240.4),
    "Qwen":  dict(EM=0.180, F1=0.226, avg_tokens=512.0),
}

rows = []
for m in MODELS:
    s = stats(traces[m])
    o = old[NICE[m]]
    rows.append({
        "Model": NICE[m],
        "EM (4.1 old)": o["EM"], "EM (4.1b)": s["EM"], "Δ EM": round(s["EM"] - o["EM"], 3),
        "F1 (4.1b)": s["F1"],
        "tokens (4.1 old)": o["avg_tokens"], "tokens (4.1b)": s["avg_tokens"],
        "N": s["N"],
    })
pd.DataFrame(rows)

,Model,EM (4.1 old),EM (4.1b),Δ EM,F1 (4.1b),tokens (4.1 old),tokens (4.1b),N
0,Phi,0.265,0.435,0.170,0.571,402.6,84.0,200
1,Gemma,0.575,0.500,-0.075,0.727,240.4,211.4,200
2,Qwen,0.180,0.620,0.440,0.773,512.0,156.0,200


**What to look for in the table above.** Two things matter:

1. **Qwen's EM should jump dramatically** — from 18% (when it was hitting the token cap on 100% of questions and never emitting an answer) to something competitive with Phi/Gemma. If it does, the `enable_thinking=False` fix worked.
2. **Phi's tokens-per-answer should drop** — the repetition-penalty fix should kill the looping.

Gemma is roughly the control: it was already running in the right mode by accident in 4.1, so its 4.1b numbers should look similar.

If those three predictions hold, the protocol fix passed sanity check, and we can trust the gate-comparison numbers in the next sections.

## 6. The four "gates" — what they look at, in plain terms

Every "step" in a trace is one chunk of generated text (~16 tokens, roughly a phrase or short sentence). At each step, we have **four detectors** voting yes/no on whether the model is about to go off-track:

| Gate | What it measures | Sentence-level analogy |
| --- | --- | --- |
| **Latent (ours)** | The hidden-state vector of the chosen middle layer suddenly shifts direction (its "drift") **and** its variance tightens (the model is locking in). | *"You're suddenly thinking more carefully — and in a new direction."* |
| **ReaLM-Retrieve** (sim) | The model's own text contains hesitation phrases like *"let me verify"*, *"actually"*, *"wait"*, *"I'm not sure"*. | *"You said the words 'wait' or 'hmm'."* |
| **GRIP** (sim) | A spike in the **entropy** of the next-word distribution — the model is suddenly less confident. | *"You're hesitating more than usual on what to say next."* |
| **Random** | A coin-flip with the same fire rate as the others, for calibration. | *"Pure chance."* |

The thresholds for each gate are tuned on the **calibration split** (120 questions) so that all gates fire on the same fraction of steps (default: 30%). This makes the comparison fair — they all "raise their hand" equally often.

Then we measure two headline quantities on the **validation split** (40 questions):

- **First-fire step** — how early does the gate raise its hand? Lower is better.
- **Trajectory drift** — across all the steps where it fires, how much is the hidden state actually changing? (We expect the latent gate to fire on big-drift steps; text-based gates to fire on whatever-step-the-word-happened.)

The Phase 4 thesis lives or dies on **mean-Δ-first-fire-step**: latent should fire on average ≥1 step earlier than ReaLM/GRIP across all three architectures.

## 7. Run the showdown

The cell below runs `scripts/04_showdown.py` against all three models. It calibrates each gate on the 120 calibration questions, then evaluates on the 40 validation questions, then writes per-model summary JSON + per-gate plots into `runs/showdown/`. Should take ~3 seconds end-to-end.

In [3]:
import subprocess, sys
out = subprocess.run(
    [sys.executable, "scripts/04_showdown.py", "--target-fire-rate", "0.30", "--suffix", "_v2"],
    cwd=ROOT, capture_output=True, text=True,
)
print(out.stdout[-2000:])
if out.returncode != 0:
    print("STDERR:", out.stderr[-1500:])

     realm_sim    79.4%   4.000   3.000   1.000      0.534   0.274   0.260
         grip_sim     82.4%   3.350   3.750  -0.400      0.500   0.321   0.179
         random       97.1%   3.174   3.400  -0.226      0.431   0.313   0.118

  [4.3]  trajectory drift (gate=latent, 5 bins by first-fire-frac)
         bin           n   ff_frac  mean_EM  mean_F1
         q1              6   0.175   0.667   0.733
         q2              6   0.244   0.333   0.611
         q3              6   0.300   0.667   0.722
         q4              6   0.385   0.833   0.833
         q5              5   1.000   1.000   1.000
         never_fired     5    n/a    0.800   0.933

  [4.4a] latent ablation: drift vs variance vs or  (only the 'latent' gate row)
         mode        fire%   ff_corr  ff_inco  delta_steps  delta_frac
         drift        85.3%   2.800   3.000  -0.200   0.213
         variance     73.5%   3.438   3.778  -0.340   0.081
         or           88.2%   2.762   2.444   0.317   0.272

  [4.5]

In [4]:
summary = json.loads((ROOT / "runs" / "phase4_showdown_v2_summary.json").read_text())["per_arch"]

def pull(m, gate):
    rows = summary[m]["earliness"]
    row = next(r for r in rows if r["gate"] == gate)
    return row

rows = []
for m in MODELS:
    lat = pull(m, "latent")
    rea = pull(m, "realm_sim")
    gri = pull(m, "grip_sim")
    ran = pull(m, "random")
    rows.append({
        "Model": NICE[m],
        "n_eligible": lat["n_eligible_traces"],
        "latent Δfrac":  round(lat["earliness_delta_frac"], 3),
        "ReaLM Δfrac":   round(rea["earliness_delta_frac"], 3),
        "GRIP Δfrac":    round(gri["earliness_delta_frac"], 3),
        "random Δfrac":  round(ran["earliness_delta_frac"], 3),
        "latent_best?":  all(lat["earliness_delta_frac"] >= x["earliness_delta_frac"] for x in (rea, gri, ran)),
    })
pd.DataFrame(rows)

,Model,n_eligible,latent Δfrac,ReaLM Δfrac,GRIP Δfrac,random Δfrac,latent_best?
0,Phi,11,-0.060,-0.009,0.224,0.180,False
1,Gemma,40,0.036,-0.130,0.015,0.041,False
2,Qwen,34,0.213,0.260,0.179,0.118,False


### How to read the table above

- **Δfrac** = the gap (in *fraction-of-trace-length* units) between when the gate first fires on **incorrect** answers vs **correct** ones. **Positive = the gate flags wrong answers earlier than right ones, which is what we want.** Higher = better.
- **latent_best?** = does the latent gate beat ReaLM, GRIP, and random simultaneously on this model.

**Honest read of the headline result (Option A protocol, fire rate = 30%):**

- **Phi (n=11)**: latent gate **loses** to GRIP and to random. *But* `n_eligible=11` is too small to draw any architectural conclusion — Phi answers in 1-2 steps on most questions, so the `min_steps≥3` filter throws away ~95% of traces. We should treat Phi's row as *inconclusive*, not as evidence against the thesis. A Phi rerun with `min_steps=2` is on the docket.
- **Gemma (n=40)**: latent gate **wins** over ReaLM (+0.036 vs −0.130), GRIP (+0.015), and random (+0.041). The win is real but narrow: latent is ~2pp better than random.
- **Qwen (n=34)**: ReaLM **narrowly beats** the latent gate by ~5pp (+0.260 vs +0.213). Latent beats GRIP and random. Interestingly, the **latent-OR ablation** (drift OR variance, see §8) flips this back to a latent win (+0.272), beating ReaLM.

**So the architecture-agnostic earliness claim does not hold cleanly at 30% fire rate.** Two productive next moves:
1. **Sweep target fire rate** down to 10% / 5%. Latent advantage usually sharpens at lower fire rates because text-based gates lose precision when forced to be selective.
2. **Adopt the latent-OR ablation as the headline gate.** It strictly dominates drift-only on every architecture. The paper text should describe this as the natural latent gate, with drift-only/variance-only as ablations.

### Trajectory drift — *what* is the latent gate locking onto?

Even if the latent gate doesn't always fire earlier than ReaLM, we can ask a different question: **does *when* the latent gate fires correlate with whether the model gets the answer right?** If the model that sets off the latent gate later in the trace is the one getting things right, then the gate is detecting *late-stage commitment* rather than early-stage drift — which is itself useful.

We sort the 40 validation traces into 5 quintiles by *when* the latent gate first fires (earliest 20% → latest 20%) and look at the mean EM/F1 in each quintile. The table below shows the earliest and latest quintile only; positive **EM gap** = "the model is more accurate on questions where the gate fires later" = the gate is finding genuinely-uncertain moments.

In [5]:
rows = []
for m in MODELS:
    bins = summary[m]["drift_latent"]
    real = [b for b in bins if b["bin"] != "never_fired" and b["mean_em"] is not None]
    if not real:
        continue
    q1, q5 = real[0], real[-1]
    rows.append({
        "Model": NICE[m],
        "fires-EARLIEST quintile EM": round(q1["mean_em"], 3),
        "fires-EARLIEST quintile F1": round(q1["mean_f1"], 3),
        "fires-LATEST  quintile EM": round(q5["mean_em"], 3),
        "fires-LATEST  quintile F1": round(q5["mean_f1"], 3),
        "EM gap (late−early)": round(q5["mean_em"] - q1["mean_em"], 3),
    })
pd.DataFrame(rows)

,Model,fires-EARLIEST quintile EM,fires-EARLIEST quintile F1,fires-LATEST quintile EM,fires-LATEST quintile F1,EM gap (late−early)
0,Phi,0.000,0.250,0.000,0.000,0.000
1,Gemma,0.500,0.779,0.625,0.708,0.125
2,Qwen,0.667,0.733,1.000,1.000,0.333


## 8. Caveats — what this Showdown does *not* prove

To stay honest, here's what we're explicitly **not** claiming yet:

1. **We didn't prove the latent gate makes the model more accurate.** We only proved it raises a flag earlier. Phase 4.6 (the *intervention* phase) is where we actually pause-and-correct the model and see if EM goes up. That's a separate experiment. Until then, "earlier flag" is a necessary-but-not-sufficient condition.
2. **The two baselines are simulators, not real implementations.** ReaLM-Retrieve and GRIP are heuristic re-implementations of the published methods, not the original code. We documented this in `_PLAN.md` (landmines L2/L3) and in the paper — readers shouldn't treat these as exact head-to-head numbers, but they're a fair "first-order" comparison.
3. **Phi traces are short.** Because Phi is small and answers fast, many of its traces are only 4-6 steps long. Earliness deltas computed on short traces are noisier. We may report Phi separately, or rerun with a lower minimum step count.
4. **Only one mid-stack layer per model.** We picked one "middle" layer based on the Phase 3 layer-sweep. The choice isn't fully ablated yet — Phase 4.5 has the multi-layer ablation on the docket.
5. **No thinking-mode arm.** Per the user's call, the headline result is the "all thinking off" arm (Option A). The "all thinking on" arm is logged as future work in `_PLAN.md` (futures F3, F4) and is the most important sensitivity test for the paper.

## 9. Verdict

The cell at the bottom prints a one-line verdict per model.

In [6]:
for m in MODELS:
    lat = pull(m, "latent"); rea = pull(m, "realm_sim"); gri = pull(m, "grip_sim"); ran = pull(m, "random")
    n = lat["n_eligible_traces"]
    score = lat["earliness_delta_frac"]
    others = {"ReaLM": rea["earliness_delta_frac"], "GRIP": gri["earliness_delta_frac"], "random": ran["earliness_delta_frac"]}
    beats = [name for name, v in others.items() if score >= v]
    if n < 20:
        verdict = "INCONCLUSIVE (too few eligible traces)"
    elif len(beats) == 3:
        verdict = "LATENT WINS (beats all baselines)"
    elif len(beats) == 2:
        verdict = f"LATENT MIXED (beats {beats}; loses to {[n for n in others if n not in beats]})"
    else:
        verdict = f"LATENT LOSES (beats only {beats})"
    cost = summary[m]["cost"]
    print(f"{NICE[m]:6} | n={n:2} EM={cost['mean_em']:.3f} | latent_Δfrac={score:+.3f} | {verdict}")

Phi    | n=11 EM=0.182 | latent_Δfrac=-0.060 | INCONCLUSIVE (too few eligible traces)
Gemma  | n=40 EM=0.525 | latent_Δfrac=+0.036 | LATENT MIXED (beats ['ReaLM', 'GRIP']; loses to ['random'])
Qwen   | n=34 EM=0.706 | latent_Δfrac=+0.213 | LATENT MIXED (beats ['GRIP', 'random']; loses to ['ReaLM'])
